# Stuck in a Gyroscope

## What kind of yard-art disaster did we unearth today

Congratulations, you've inherited a neural network that trains without crashing, prints tidy little epoch logs, and produces test accuracy that would embarrass a coin flip. That's the scary part. This isn't a stack trace begging for help. It's a model that *looks* fine, sounds fine, and is, in fact, deeply, quietly not fine.

Run it. Watch the loss wiggle downward like it's trying. Watch the final accuracy land somewhere in "might as well have guessed" territory. Somewhere in this network, a perfectly good idea got pointed in the wrong direction and it's still standing there, confidently, doing math in the wrong direction. Like a gyroscope spinning beautifully around an axis nobody asked for.

I'm not going to tell you where. I'm going to tell you *what*, in excruciating detail, and then I'm going to watch you find it yourself. That's the deal.

Expected symptom when run as-is: training does not crash and the loss even wiggles downward a little, but it plateaus early and test accuracy sits stubbornly near chance level (roughly 10-20% on 10-class MNIST) instead of climbing toward the 90%+ you'd expect from this architecture. 

## Run It

Drop the four dataset files into `dataset/`, then run `python gyroscope.py` (or run every cell below, top to bottom).

*Data loading. Riveting stuff. Nobody has ever broken a model with a data loader.... allegedly.*

In [4]:
import numpy as np
import struct
import os

DATA_DIR = "dataset"


def load_idx_images(path):
    """Load MNIST image file in idx3-ubyte format."""
    with open(path, "rb") as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        buf = f.read(rows * cols * num)
        images = np.frombuffer(buf, dtype=np.uint8).reshape(num, rows * cols)
    return images


def load_idx_labels(path):
    """Load MNIST label file in idx1-ubyte format."""
    with open(path, "rb") as f:
        magic, num = struct.unpack(">II", f.read(8))
        buf = f.read(num)
        labels = np.frombuffer(buf, dtype=np.uint8)
    return labels


def load_dataset():
    """Load train/test MNIST splits from disk."""
    X_train = load_idx_images(os.path.join(DATA_DIR, "train-images.idx3-ubyte"))
    y_train = load_idx_labels(os.path.join(DATA_DIR, "train-labels.idx1-ubyte"))
    X_test = load_idx_images(os.path.join(DATA_DIR, "t10k-images.idx3-ubyte"))
    y_test = load_idx_labels(os.path.join(DATA_DIR, "t10k-labels.idx1-ubyte"))
    return X_train, y_train, X_test, y_test


*Normalize the pixels, one-hot the labels. If you're going to fail, at least fail on properly scaled inputs.*

In [5]:
def preprocess(X, y, num_classes=10):
    """Normalize pixel values and one-hot encode labels."""
    X = X.astype(np.float64) / 255.0
    y_onehot = np.zeros((y.shape[0], num_classes))
    y_onehot[np.arange(y.shape[0]), y] = 1
    return X, y_onehot


*He initialization, because apparently zeros are for cowards and NaNs are for people who pair zeros with ReLU.*

No not the pronoun "He". God, just look up Kaiming He

In [6]:
def init_params(input_dim=784, hidden_dim=128, output_dim=10, seed=42):
    """Initialize weights and biases for a 2-layer MLP."""
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, np.sqrt(2.0 / input_dim), size=(input_dim, hidden_dim))
    b1 = np.zeros((1, hidden_dim))
    W2 = rng.normal(0, np.sqrt(2.0 / hidden_dim), size=(hidden_dim, output_dim))
    b2 = np.zeros((1, output_dim))
    return W1, b1, W2, b2


*Two activation functions. One of them is about to ruin your week. I'm not saying which. I already told you how to figure it out.*

In [ ]:
def relu(z):
    """Standard ReLU activation."""
    return np.maximum(0, z)


def relu_grad(z):
    """Gradient of ReLU."""
    return (z > 0).astype(z.dtype)


def softmax(z):
    """Convert a batch of logits into probability distributions."""
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


*The forward pass, assembled the usual way. Nothing to see here*

In [8]:
def forward(X, W1, b1, W2, b2):
    """Run a forward pass through the network."""
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)
    cache = (Z1, A1, Z2, A2)
    return A2, cache


*Cross-entropy loss. Expects actual per-example probabilities. Manages its expectations poorly.*

In [9]:
def cross_entropy_loss(probs, y_onehot):
    """Compute average cross-entropy loss over a batch."""
    m = y_onehot.shape[0]
    correct_probs = np.sum(probs * y_onehot, axis=1)
    log_likelihood = -np.log(correct_probs + 1e-9)
    return np.sum(log_likelihood) / m


*Backprop, straight from the textbook.*

In [10]:
def backward(X, y_onehot, cache, W2):
    """Backpropagate gradients through the network."""
    Z1, A1, Z2, A2 = cache
    m = X.shape[0]
    dZ2 = (A2 - y_onehot) / m
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_grad(Z1)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2


def update_params(params, grads, lr=0.1):
    """Apply a gradient descent step."""
    W1, b1, W2, b2 = params
    dW1, db1, dW2, db2 = grads
    W1 = W1 - lr * dW1
    b1 = b1 - lr * db1
    W2 = W2 - lr * dW2
    b2 = b2 - lr * db2
    return W1, b1, W2, b2


def accuracy(probs, y_onehot):
    """Compute classification accuracy."""
    preds = np.argmax(probs, axis=1)
    labels = np.argmax(y_onehot, axis=1)
    return np.mean(preds == labels)


*The training loop. Watch it try its best. It's honestly kind of sweet.*

In [11]:
def train(X_train, y_train, X_test, y_test, epochs=15, batch_size=128, lr=0.1):
    """Train the network with mini-batch gradient descent."""
    W1, b1, W2, b2 = init_params()
    n = X_train.shape[0]

    for epoch in range(epochs):
        perm = np.random.permutation(n)
        X_shuf, y_shuf = X_train[perm], y_train[perm]

        for i in range(0, n, batch_size):
            X_batch = X_shuf[i:i + batch_size]
            y_batch = y_shuf[i:i + batch_size]

            probs, cache = forward(X_batch, W1, b1, W2, b2)
            grads = backward(X_batch, y_batch, cache, W2)
            W1, b1, W2, b2 = update_params((W1, b1, W2, b2), grads, lr)

        train_probs, _ = forward(X_train, W1, b1, W2, b2)
        loss = cross_entropy_loss(train_probs, y_train)
        acc = accuracy(train_probs, y_train)
        print(f"Epoch {epoch+1:2d}  loss={loss:.4f}  train_acc={acc:.4f}")

    test_probs, _ = forward(X_test, W1, b1, W2, b2)
    test_acc = accuracy(test_probs, y_test)
    print(f"\nFinal test accuracy: {test_acc:.4f}")
    return W1, b1, W2, b2


In [ ]:
if __name__ == "__main__":
    X_train, y_train_raw, X_test, y_test_raw = load_dataset()
    X_train, y_train = preprocess(X_train, y_train_raw)
    X_test, y_test = preprocess(X_test, y_test_raw)
    train(X_train, y_train, X_test, y_test)


Epoch  1  loss=20.7229  train_acc=0.0987
Epoch  2  loss=20.7229  train_acc=0.0987
Epoch  3  loss=20.7229  train_acc=0.0987
Epoch  4  loss=20.7229  train_acc=0.0987


/tmp/ipykernel_35384/481896119.py:5: RuntimeWarning: overflow encountered in matmul
  Z2 = A1 @ W2 + b2
/tmp/ipykernel_35384/395604252.py:13: RuntimeWarning: invalid value encountered in subtract
  z_shift = z - np.max(z, axis=0, keepdims=True)


Epoch  5  loss=nan  train_acc=0.0987
Epoch  6  loss=nan  train_acc=0.0987
Epoch  7  loss=nan  train_acc=0.0987
Epoch  8  loss=nan  train_acc=0.0987
Epoch  9  loss=nan  train_acc=0.0987
Epoch 10  loss=nan  train_acc=0.0987
Epoch 11  loss=nan  train_acc=0.0987
Epoch 12  loss=nan  train_acc=0.0987
Epoch 13  loss=nan  train_acc=0.0987
Epoch 14  loss=nan  train_acc=0.0987
Epoch 15  loss=nan  train_acc=0.0987

Final test accuracy: 0.0980


## The Math Clue (yes, I'm just going to hand this to you)

Softmax turns logits into probabilities: softmax(z)_i = exp(z_i) / Σ exp(z_j). Two requirements: outputs are positive, and they sum to 1 per example. The max-subtraction trick (exp(z_i − max z)) is just for numerical stability. Same math, no overflow. Worth checking this function out... there's a mistake kinda very ish here. But not here hehe :P

I guess gyroscope is kind of a clue ? Idk but it should click once you solve it 

## Submission

Fork this repo → fix the bug → open a PR → wait for a human to confirm the fix is correct and the symptom is resolved → receive your title.

## Your Title, Should You Succeed

**"Grand Axis Whisperer, Reconciler of Rows and Columns, Third Chair of the Tensor Orientation Tribunal"**

*Oh ma god you found it. You're no longer stuck in a gyroscope. Congratualations!* Wipes a tear off face. *I'm such an amazing teacher*. Be proud of that title and for the love of god do not make a mistake like this in your life